# 03 — Feature Engineering

Input: `nlp_preprocessed_recipes.parquet` from `02_nlp_preprocessing.ipynb`.

Outputs:
- SBERT (all-MiniLM-L6-v2) embeddings for `combined_text`
- Scaled nutrition features (Food.com rows only — `nutrition_available` flag handles the rest)
- FAISS index over the embeddings for fast similarity search

Confirmed: <15% of recipes exceed MiniLM's 256-token limit, so proceeding with MiniLM as-is.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
import numpy as np

In [ ]:
PROCESSED_DIR = Path.cwd().parent / "datasets" / "processed"

df = pd.read_parquet(PROCESSED_DIR / "nlp_preprocessed_recipes.parquet")
print(df.shape)
df[["title", "combined_text", "nutrition_available"]].head(3)

## 1. Install / check dependencies

Run once if not already installed:
```bash
pip install sentence-transformers faiss-cpu
```
`faiss-cpu` — you're not doing GPU-scale search here, CPU is fine and avoids CUDA setup headaches.

## 2. Generate SBERT embeddings

Batched encoding — do not loop row-by-row, `model.encode()` batches internally and is dramatically faster.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = df["combined_text"].tolist()

embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # L2-normalize so cosine similarity == dot product (needed for FAISS IndexFlatIP)
)

print(embeddings.shape)  # (n_recipes, 384) for MiniLM-L6-v2

In [ ]:
EMBEDDINGS_DIR = Path.cwd().parent / "datasets" / "embeddings"
EMBEDDINGS_DIR.mkdir(parents=True, exist_ok=True)

np.save(EMBEDDINGS_DIR / "sbert_embeddings.npy", embeddings)
print(f"Saved embeddings: {embeddings.shape} -> {EMBEDDINGS_DIR / 'sbert_embeddings.npy'}")

## 3. Nutrition features

Scale only the rows where `nutrition_available == True`. Do not impute nutrition for RecipeNLG rows with fake/mean values — that would fabricate signal that doesn't exist and quietly bias the hybrid ranker later. Keep NaN, handle it explicitly downstream.

In [ ]:
from sklearn.preprocessing import StandardScaler

nutrition_cols = ["calories", "total_fat_pdv", "sugar_pdv", "sodium_pdv",
                   "protein_pdv", "sat_fat_pdv", "carbs_pdv"]

has_nutrition = df["nutrition_available"] == True
print(f"Rows with nutrition: {has_nutrition.sum()} / {len(df)} ({has_nutrition.mean()*100:.1f}%)")

scaler = StandardScaler()
nutrition_scaled = np.full((len(df), len(nutrition_cols)), np.nan)
nutrition_scaled[has_nutrition.values] = scaler.fit_transform(df.loc[has_nutrition, nutrition_cols])

nutrition_scaled_df = pd.DataFrame(
    nutrition_scaled, columns=[f"{c}_scaled" for c in nutrition_cols], index=df.index
)
nutrition_scaled_df.head()

In [ ]:
import joblib

MODELS_DIR = Path.cwd().parent / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(scaler, MODELS_DIR / "nutrition_scaler.pkl")
print(f"Saved scaler -> {MODELS_DIR / 'nutrition_scaler.pkl'}")

## 4. Build FAISS index

`IndexFlatIP` (inner product) on L2-normalized embeddings = exact cosine similarity search. Exact, not approximate — fine at this dataset size. Switch to `IndexIVFFlat` only if search latency becomes a real problem at serving time; don't add that complexity preemptively.

In [ ]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print(f"FAISS index built: {index.ntotal} vectors, dim={dimension}")

In [ ]:
faiss.write_index(index, str(EMBEDDINGS_DIR / "faiss_index.bin"))
print(f"Saved FAISS index -> {EMBEDDINGS_DIR / 'faiss_index.bin'}")

## 5. Sanity check — does similarity search actually make sense?

Don't skip this. Pick a recipe, retrieve its nearest neighbors, and eyeball whether they're actually similar. If this looks wrong, something upstream (text combination, truncation, dedup) is broken — better to catch it now than after building the ranker on top of bad embeddings.

In [ ]:
query_idx = 0
print("Query recipe:", df.iloc[query_idx]["title"])

k = 6  # top 5 + itself
query_vec = embeddings[query_idx:query_idx+1]
distances, indices = index.search(query_vec, k)

print("\nNearest neighbors:")
for rank, (idx, score) in enumerate(zip(indices[0], distances[0])):
    print(f"{rank}. {df.iloc[idx]['title']}  (score={score:.3f})")

## 6. Save final feature table

Combine everything into one dataframe aligned by row index with the saved `.npy`/FAISS files. Embeddings themselves stay in `.npy` (not stuffed into the dataframe) — keeps the parquet lightweight and avoids re-serializing 384-dim arrays through pandas.

In [ ]:
features_df = pd.concat([df.reset_index(drop=True), nutrition_scaled_df.reset_index(drop=True)], axis=1)

features_df.to_parquet(PROCESSED_DIR / "feature_engineered_recipes.parquet", index=False)
print(f"Saved {len(features_df)} rows to {PROCESSED_DIR / 'feature_engineered_recipes.parquet'}")
print("\nRow order in this parquet matches row order in sbert_embeddings.npy and the FAISS index — keep it that way.")

## Next notebook: `04_recommenders.ipynb`

Content-based (KNN/cosine via this FAISS index) first, validate it works standalone, then collaborative filtering (SVD/LightFM) — but check Food.com interaction data density before building the collaborative branch, per the earlier discussion. If interactions are sparse, that changes how much weight collaborative filtering should get in the hybrid ranker.